# PySpark practice 4: functions for strings, math and conditions

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-practice-04-functions-strings-and-math.ipynb)

17 exercises on conditions, predicates, strings and regex, rounding and safe math, and hashing. Each exercise names the [reference entries](pyspark-reference-04-functions-strings-and-math.ipynb) it practises, shows its inputs and the exact expected output, and hides a reference solution you can reveal when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local session, the shared `customers`, `orders`, `order_items`, `products` and `events` tables (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry), and the helpers from [PySpark reference 4](pyspark-reference-04-functions-strings-and-math.ipynb#setup-and-sample-data). Exercises that need other data create it in their input cell.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import Window
from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

products = spark.createDataFrame([
    ("SKU-4001", "Kitchen", D("12.50")), ("SKU-4002", "Kitchen", D("34.50")),
    ("SKU-4003", "Garden", D("149.00")), ("SKU-4004", "Garden", D("55.50")),
    ("SKU-4005", "Pantry", D("55.25")), ("SKU-4006", "Outdoor", D("189.00")),
    ("SKU-4007", "Kitchen", D("95.00")), ("SKU-4008", "Outdoor", D("121.00")),
    ("SKU-4009", "Pantry", D("25.00")), ("SKU-4010", "Garden", D("89.90")),
    ("SKU-4011", "Pantry", D("59.00")), ("SKU-4012", "Kitchen", D("37.00")),
    ("SKU-4013", "Outdoor", D("210.00")),  # SKU-4014 is missing
], "sku string, category string, list_price decimal(8,2)")

events = spark.createDataFrame([
    (1, "view", datetime(2026, 1, 6, 9, 0)), (1, "add_to_cart", datetime(2026, 1, 6, 9, 2)),
    (1, "purchase", datetime(2026, 1, 6, 9, 10)), (2, "view", datetime(2026, 1, 7, 10, 0)),
    (2, "purchase", datetime(2026, 1, 7, 10, 40)), (3, "view", datetime(2026, 1, 6, 11, 15)),
    (3, "view", datetime(2026, 1, 8, 11, 20)), (4, "add_to_cart", datetime(2026, 1, 9, 20, 5)),
    (4, "purchase", datetime(2026, 1, 10, 8, 30)), (5, "view", datetime(2026, 1, 11, 14, 45)),
    (5, "view", datetime(2026, 1, 12, 9, 0)), (5, "purchase", datetime(2026, 1, 12, 12, 30)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),  # an SDK retry committed this event twice
], "user_id int, event string, ts timestamp")

import contextlib
import io
import logging
import re


def plan_lines(df, mode="simple"):
    """The lines df.explain(mode) prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue().splitlines()


def final_plan(df):
    """Run df, then return the plan Spark actually used (AQE's final plan when there is one)."""
    df.collect()
    text = "\n".join(plan_lines(df))
    if "== Final Plan ==" in text:
        text = text.split("== Final Plan ==")[1].split("== Initial Plan ==")[0]
    return text


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        print(type(error).__name__ + ":", str(error).split(".")[0][:100])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

## Exercises

9 easy, 7 medium, 1 hard. Write your answer in the empty cell under each exercise as a DataFrame called `result`, run it, and compare it with the expected output (columns, values and row order).

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [Normalize emails into a join key](#exercise-1-normalize-emails-into-a-join-key) | Easy | [`trim`, `ltrim`, `rtrim`](pyspark-reference-04-functions-strings-and-math.ipynb#trim-ltrim-rtrim), [`upper`, `lower`, `initcap`](pyspark-reference-04-functions-strings-and-math.ipynb#upper-lower-initcap) |
| 2 | [Extract parts of a location code](#exercise-2-extract-parts-of-a-location-code) | Medium | [`regexp_extract`](pyspark-reference-04-functions-strings-and-math.ipynb#regexp_extract) |
| 3 | [Mask personal data](#exercise-3-mask-personal-data) | Medium | [`regexp_replace`](pyspark-reference-04-functions-strings-and-math.ipynb#regexp_replace) |
| 4 | [Split a category path](#exercise-4-split-a-category-path) | Easy | [`split`, `split_part`, `substring_index`](pyspark-reference-04-functions-strings-and-math.ipynb#split-split_part-substring_index) |
| 5 | [Build zero-padded labels](#exercise-5-build-zero-padded-labels) | Easy | [`format_string`, `format_number`](pyspark-reference-04-functions-strings-and-math.ipynb#format_string-format_number), [`lpad`, `rpad`](pyspark-reference-04-functions-strings-and-math.ipynb#lpad-rpad) |
| 6 | [Strip a phone number to digits](#exercise-6-strip-a-phone-number-to-digits) | Easy | [`translate`, `replace`](pyspark-reference-04-functions-strings-and-math.ipynb#translate-replace) |
| 7 | [Tier orders with ordered conditions](#exercise-7-tier-orders-with-ordered-conditions) | Easy | [`when` / `otherwise`](pyspark-reference-04-functions-strings-and-math.ipynb#when--otherwise) |
| 8 | [Pick the preferred contact channel](#exercise-8-pick-the-preferred-contact-channel) | Easy | [`coalesce`](pyspark-reference-04-functions-strings-and-math.ipynb#coalesce), [`nullif`](pyspark-reference-04-functions-strings-and-math.ipynb#nullif) |
| 9 | [Validate SKU formats](#exercise-9-validate-sku-formats) | Medium | [`rlike`](pyspark-reference-04-functions-strings-and-math.ipynb#rlike), [`like`, `ilike`](pyspark-reference-04-functions-strings-and-math.ipynb#like-ilike) |
| 10 | [Round money with the right mode](#exercise-10-round-money-with-the-right-mode) | Medium | [`round`, `bround`](pyspark-reference-04-functions-strings-and-math.ipynb#round-bround) |
| 11 | [Assign stable buckets](#exercise-11-assign-stable-buckets) | Medium | [`pmod`](pyspark-reference-04-functions-strings-and-math.ipynb#pmod), [`hash`, `xxhash64`](pyspark-reference-04-functions-strings-and-math.ipynb#hash-xxhash64) |
| 12 | [Clamp values into a valid range](#exercise-12-clamp-values-into-a-valid-range) | Easy | [`greatest`, `least`](pyspark-reference-04-functions-strings-and-math.ipynb#greatest-least) |
| 13 | [Surrogate keys that cannot collide on concatenation](#exercise-13-surrogate-keys-that-cannot-collide-on-concatenation) | Hard | [`md5`, `sha2`](pyspark-reference-04-functions-strings-and-math.ipynb#md5-sha2) |
| 14 | [Pseudonymize emails for sharing](#exercise-14-pseudonymize-emails-for-sharing) | Medium | [`md5`, `sha2`](pyspark-reference-04-functions-strings-and-math.ipynb#md5-sha2) |
| 15 | [Safe ratios per customer](#exercise-15-safe-ratios-per-customer) | Medium | [`try_add`, `try_subtract`, `try_multiply`, `try_divide`](pyspark-reference-04-functions-strings-and-math.ipynb#try_add-try_subtract-try_multiply-try_divide) |
| 16 | [Concatenate a display name around nulls](#exercise-16-concatenate-a-display-name-around-nulls) | Easy | [`concat`, `concat_ws`](pyspark-reference-04-functions-strings-and-math.ipynb#concat-concat_ws) |
| 17 | [Filter with membership and ranges](#exercise-17-filter-with-membership-and-ranges) | Easy | [`isin`, `between`](pyspark-reference-04-functions-strings-and-math.ipynb#isin-between) |

## Exercise 1: Normalize emails into a join key

**Difficulty:** Easy · about 5 min · **Practises:** [`trim`, `ltrim`, `rtrim`](pyspark-reference-04-functions-strings-and-math.ipynb#trim-ltrim-rtrim), [`upper`, `lower`, `initcap`](pyspark-reference-04-functions-strings-and-math.ipynb#upper-lower-initcap)

*Matching customers across systems starts with one canonical key.*

Return `id` and `email_key`: the email trimmed and lower-cased, null when the result is empty. Order by `id`.

In [ ]:
raw = spark.createDataFrame([(1, "  Ada@Example.COM "), (2, "grace@example.org"), (3, "   "), (4, None)], "id int, email string")
raw.show(truncate=False)

**Expected output**

| id | email_key |
| --- | --- |
| 1 | ada@example.com |
| 2 | grace@example.org |
| 3 | null |
| 4 | null |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = raw.select("id", F.nullif(F.lower(F.trim("email")), F.lit("")).alias("email_key")).orderBy("id")
result.show()

## Exercise 2: Extract parts of a location code

**Difficulty:** Medium · about 10 min · **Practises:** [`regexp_extract`](pyspark-reference-04-functions-strings-and-math.ipynb#regexp_extract)

*Compound codes carry country, city and store number in one string.*

Codes look like `UK-LON-0042`. Return `code`, `country`, `city` and `store` (as int), with nulls for codes that do not match the pattern. Keep input order.

In [ ]:
codes = spark.createDataFrame([(1, "UK-LON-0042"), (2, "US-NYC-0007"), (3, "unassigned"), (4, "DE-BER-12")], "n int, code string")
codes.show()

**Expected output**

| code | country | city | store |
| --- | --- | --- | --- |
| UK-LON-0042 | UK | LON | 42 |
| US-NYC-0007 | US | NYC | 7 |
| unassigned | null | null | null |
| DE-BER-12 | null | null | null |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
pattern = r"^([A-Z]{2})-([A-Z]{3})-(\d{4})$"
part = lambda i: F.nullif(F.regexp_extract("code", pattern, i), F.lit(""))
result = codes.orderBy("n").select("code", part(1).alias("country"), part(2).alias("city"), part(3).cast("int").alias("store"))
result.show()

## Exercise 3: Mask personal data

**Difficulty:** Medium · about 10 min · **Practises:** [`regexp_replace`](pyspark-reference-04-functions-strings-and-math.ipynb#regexp_replace)

*Logs and extracts shared outside the team must not expose emails or card numbers.*

Mask the user part of each email after its first character (`a***@example.com`) and every digit of the card number except the last four (`************4242`). Return `id`, `email_masked`, `card_masked`, ordered by `id`.

In [ ]:
pii = spark.createDataFrame([(1, "ada@example.com", "4242424242424242"), (2, "grace.h@corp.io", "5555555555554444")], "id int, email string, card string")
pii.show(truncate=False)

**Expected output**

| id | email_masked | card_masked |
| --- | --- | --- |
| 1 | a***@example.com | ************4242 |
| 2 | g***@corp.io | ************4444 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = pii.select(
    "id",
    F.regexp_replace("email", r"^(.)[^@]*@", "$1***@").alias("email_masked"),
    F.regexp_replace("card", r"\d(?=\d{4})", "*").alias("card_masked"),
).orderBy("id")
result.show(truncate=False)

## Exercise 4: Split a category path

**Difficulty:** Easy · about 5 min · **Practises:** [`split`, `split_part`, `substring_index`](pyspark-reference-04-functions-strings-and-math.ipynb#split-split_part-substring_index)

*Hierarchies arrive flattened into delimited strings.*

Paths use `>` between levels. Return `path`, `top` (first level), `leaf` (last level) and `depth` (number of levels). Keep input order.

In [ ]:
paths = spark.createDataFrame([(1, "Kitchen>Knives>Chef"), (2, "Garden>Hoses"), (3, "Pantry")], "n int, path string")
paths.show(truncate=False)

**Expected output**

| path | top | leaf | depth |
| --- | --- | --- | --- |
| Kitchen>Knives>Chef | Kitchen | Chef | 3 |
| Garden>Hoses | Garden | Hoses | 2 |
| Pantry | Pantry | Pantry | 1 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = paths.orderBy("n").select(
    "path",
    F.split_part("path", F.lit(">"), F.lit(1)).alias("top"),
    F.split_part("path", F.lit(">"), F.lit(-1)).alias("leaf"),
    F.size(F.split("path", ">")).alias("depth"),
)
result.show(truncate=False)

## Exercise 5: Build zero-padded labels

**Difficulty:** Easy · about 5 min · **Practises:** [`format_string`, `format_number`](pyspark-reference-04-functions-strings-and-math.ipynb#format_string-format_number), [`lpad`, `rpad`](pyspark-reference-04-functions-strings-and-math.ipynb#lpad-rpad)

*Invoice and shipment numbers have fixed formats downstream systems validate.*

Return `order_id`, `invoice` (`INV-2026-` followed by the order id padded to 5 digits) and `amount_text` (amount with thousands separators and 2 decimals) for orders 8 to 10, ordered by `order_id`.

In [ ]:
orders.select("order_id", "amount").show()

**Expected output**

| order_id | invoice | amount_text |
| --- | --- | --- |
| 8 | INV-2026-00008 | 89.90 |
| 9 | INV-2026-00009 | 1,180.00 |
| 10 | INV-2026-00010 | 74.00 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = orders.where("order_id BETWEEN 8 AND 10").select(
    "order_id", F.format_string("INV-2026-%05d", "order_id").alias("invoice"), F.format_number("amount", 2).alias("amount_text")
).orderBy("order_id")
result.show()

## Exercise 6: Strip a phone number to digits

**Difficulty:** Easy · about 5 min · **Practises:** [`translate`, `replace`](pyspark-reference-04-functions-strings-and-math.ipynb#translate-replace)

*Telephony systems need E.164-style digits only.*

Return `id` and `digits` (the phone with every non-digit removed). Order by `id`.

In [ ]:
phones = spark.createDataFrame([(1, "(512) 555-0100"), (2, "+44 20 7946 0958"), (3, "555.0199")], "id int, phone string")
phones.show()

**Expected output**

| id | digits |
| --- | --- |
| 1 | 5125550100 |
| 2 | 442079460958 |
| 3 | 5550199 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = phones.select("id", F.regexp_replace("phone", r"\D", "").alias("digits")).orderBy("id")
result.show()

## Exercise 7: Tier orders with ordered conditions

**Difficulty:** Easy · about 5 min · **Practises:** [`when` / `otherwise`](pyspark-reference-04-functions-strings-and-math.ipynb#when--otherwise)

*Business tiers depend on testing the most specific rule first.*

Return `order_id` and `tier`: `enterprise` for amounts of 1,000 or more, `mid_market` for 100 or more, `small` otherwise, and `review` for any `returned` order regardless of amount. Order by `order_id`.

In [ ]:
orders.select("order_id", "status", "amount").show()

**Expected output**

| order_id | tier |
| --- | --- |
| 1 | mid_market |
| 2 | small |
| 3 | review |
| 4 | mid_market |
| 5 | small |
| 6 | mid_market |
| 7 | mid_market |
| 8 | review |
| 9 | enterprise |
| 10 | small |
| 11 | mid_market |
| 12 | small |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = orders.select(
    "order_id",
    F.when(F.col("status") == "returned", "review")
     .when(F.col("amount") >= 1000, "enterprise")
     .when(F.col("amount") >= 100, "mid_market")
     .otherwise("small").alias("tier"),
).orderBy("order_id")
result.show()

## Exercise 8: Pick the preferred contact channel

**Difficulty:** Easy · about 5 min · **Practises:** [`coalesce`](pyspark-reference-04-functions-strings-and-math.ipynb#coalesce), [`nullif`](pyspark-reference-04-functions-strings-and-math.ipynb#nullif)

*Customer records hold several optional contact fields of uneven quality.*

Return `id` and `contact`: the mobile number, else the home number, else the email, else `none`; blank strings count as missing. Order by `id`.

In [ ]:
people = spark.createDataFrame([(1, "555-01", None, "a@x.com"), (2, " ", "555-02", None), (3, None, None, "c@x.com"), (4, "", None, None)],
                               "id int, mobile string, home string, email string")
people.show()

**Expected output**

| id | contact |
| --- | --- |
| 1 | 555-01 |
| 2 | 555-02 |
| 3 | c@x.com |
| 4 | none |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
blank_to_null = lambda c: F.nullif(F.trim(c), F.lit(""))
result = people.select("id", F.coalesce(blank_to_null("mobile"), blank_to_null("home"), blank_to_null("email"), F.lit("none")).alias("contact")).orderBy("id")
result.show()

## Exercise 9: Validate SKU formats

**Difficulty:** Medium · about 10 min · **Practises:** [`rlike`](pyspark-reference-04-functions-strings-and-math.ipynb#rlike), [`like`, `ilike`](pyspark-reference-04-functions-strings-and-math.ipynb#like-ilike)

*Reject malformed product codes before they reach the catalog.*

Return `sku`, `valid` (exactly `SKU-` followed by 4 digits, case-sensitive) and `looks_like_sku` (starts with `sku-` in any case). Keep input order.

In [ ]:
skus = spark.createDataFrame([(1, "SKU-4001"), (2, "sku-4002"), (3, "SKU-40"), (4, "SKU-4003x"), (5, "ITEM-1")], "n int, sku string")
skus.show()

**Expected output**

| sku | valid | looks_like_sku |
| --- | --- | --- |
| SKU-4001 | true | true |
| sku-4002 | false | true |
| SKU-40 | false | true |
| SKU-4003x | false | true |
| ITEM-1 | false | false |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = skus.orderBy("n").select("sku", F.col("sku").rlike(r"^SKU-\d{4}$").alias("valid"), F.col("sku").ilike("sku-%").alias("looks_like_sku"))
result.show()

## Exercise 10: Round money with the right mode

**Difficulty:** Medium · about 10 min · **Practises:** [`round`, `bround`](pyspark-reference-04-functions-strings-and-math.ipynb#round-bround)

*Reporting rules specify half-up or banker's rounding; the totals differ.*

For `values`, return `value`, `half_up` (`round` to 0 decimals) and `half_even` (`bround`), then a final row `TOTAL` with the sum of each column. Use string `value` labels.

In [ ]:
values = spark.createDataFrame([(D("0.5"),), (D("1.5"),), (D("2.5"),), (D("3.5"),)], "value decimal(4,1)")
values.show()

**Expected output**

| value | half_up | half_even |
| --- | --- | --- |
| 0.5 | 1 | 0 |
| 1.5 | 2 | 2 |
| 2.5 | 3 | 2 |
| 3.5 | 4 | 4 |
| TOTAL | 10 | 8 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
rounded = values.select(F.col("value").cast("string").alias("value"), F.round("value").alias("half_up"), F.bround("value").alias("half_even"))
totals = rounded.agg(F.lit("TOTAL").alias("value"), F.sum("half_up").alias("half_up"), F.sum("half_even").alias("half_even"))
result = (
    rounded.withColumn("is_total", F.lit(0)).unionByName(totals.withColumn("is_total", F.lit(1)))
    .orderBy("is_total", "value").drop("is_total")
)
result.show()

## Exercise 11: Assign stable buckets

**Difficulty:** Medium · about 10 min · **Practises:** [`pmod`](pyspark-reference-04-functions-strings-and-math.ipynb#pmod), [`hash`, `xxhash64`](pyspark-reference-04-functions-strings-and-math.ipynb#hash-xxhash64)

*A/B tests and sharding need the same bucket for the same key on every run.*

Assign each customer to one of 4 buckets with `pmod(xxhash64(customer_id), 4)`. Return `customer_id` and `bucket`, ordered by `customer_id`.

In [ ]:
customers.select("customer_id").show()

**Expected output**

| customer_id | bucket |
| --- | --- |
| 1 | 1 |
| 2 | 2 |
| 3 | 3 |
| 4 | 0 |
| 5 | 0 |
| 6 | 1 |
| 7 | 3 |
| 8 | 3 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = customers.select("customer_id", F.pmod(F.xxhash64("customer_id"), F.lit(4)).alias("bucket")).orderBy("customer_id")
result.show()

## Exercise 12: Clamp values into a valid range

**Difficulty:** Easy · about 5 min · **Practises:** [`greatest`, `least`](pyspark-reference-04-functions-strings-and-math.ipynb#greatest-least)

*Sensor and score values outside the valid range are clipped, not dropped.*

Clamp `score` into 0–100, keeping nulls as null. Return `id` and `clamped`, ordered by `id`.

In [ ]:
scores = spark.createDataFrame([(1, -5), (2, 42), (3, 130), (4, None)], "id int, score int")
scores.show()

**Expected output**

| id | clamped |
| --- | --- |
| 1 | 0 |
| 2 | 42 |
| 3 | 100 |
| 4 | null |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = scores.select("id", F.when(F.col("score").isNotNull(), F.least(F.greatest("score", F.lit(0)), F.lit(100))).alias("clamped")).orderBy("id")
result.show()

## Exercise 13: Surrogate keys that cannot collide on concatenation

**Difficulty:** Hard · about 15 min · **Practises:** [`md5`, `sha2`](pyspark-reference-04-functions-strings-and-math.ipynb#md5-sha2)

*Hashing `concat(a, b)` makes ('ab','c') and ('a','bc') the same key.*

For each row return `id`, `naive_key` (first 8 characters of `md5(concat(a, b))`) and `safe_key` (first 8 characters of `md5(concat_ws('|', a, b))`), ordered by `id`. Compare the two keys across rows.

In [ ]:
parts = spark.createDataFrame([(1, "ab", "c"), (2, "a", "bc")], "id int, a string, b string")
parts.show()

**Expected output**

| id | naive_key | safe_key |
| --- | --- | --- |
| 1 | 90015098 | 75edb597 |
| 2 | 90015098 | 8a7d22c8 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = parts.select(
    "id",
    F.substring(F.md5(F.concat("a", "b")), 1, 8).alias("naive_key"),
    F.substring(F.md5(F.concat_ws("|", "a", "b")), 1, 8).alias("safe_key"),
).orderBy("id")
result.show()

## Exercise 14: Pseudonymize emails for sharing

**Difficulty:** Medium · about 10 min · **Practises:** [`md5`, `sha2`](pyspark-reference-04-functions-strings-and-math.ipynb#md5-sha2)

*Analytics extracts replace identifiers with stable salted hashes.*

Return `id` and `email_hash`: the first 16 characters of `sha2(salt || lower(trim(email)), 256)` with salt `s3cret`. The same email in different casing must get the same hash. Order by `id`.

In [ ]:
emails = spark.createDataFrame([(1, "Ada@Example.com"), (2, " ada@example.com "), (3, "grace@example.org")], "id int, email string")
emails.show()

**Expected output**

| id | email_hash |
| --- | --- |
| 1 | 4e5860a3440c23ee |
| 2 | 4e5860a3440c23ee |
| 3 | 313e2f5cc9571890 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = emails.select("id", F.substring(F.sha2(F.concat(F.lit("s3cret"), F.lower(F.trim("email"))), 256), 1, 16).alias("email_hash")).orderBy("id")
result.show(truncate=False)

## Exercise 15: Safe ratios per customer

**Difficulty:** Medium · about 10 min · **Practises:** [`try_add`, `try_subtract`, `try_multiply`, `try_divide`](pyspark-reference-04-functions-strings-and-math.ipynb#try_add-try_subtract-try_multiply-try_divide)

*Rates with zero denominators must not fail the report.*

For each customer (all 8), return `customer_id` and `return_rate`: returned orders divided by all orders, rounded to 2 decimals, null for customers without orders. Order by `customer_id`.

In [ ]:
orders.select("order_id", "customer_id", "status").show()

**Expected output**

| customer_id | return_rate |
| --- | --- |
| 1 | 0.33 |
| 2 | 0.0 |
| 3 | 0.0 |
| 4 | 0.5 |
| 5 | 0.0 |
| 6 | 0.0 |
| 7 | null |
| 8 | null |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
counts = customers.join(orders, "customer_id", "left").groupBy("customer_id").agg(
    F.count("order_id").alias("orders"), F.count_if(F.col("status") == "returned").alias("returns"))
result = counts.select("customer_id", F.round(F.try_divide("returns", "orders"), 2).alias("return_rate")).orderBy("customer_id")
result.show()

## Exercise 16: Concatenate a display name around nulls

**Difficulty:** Easy · about 5 min · **Practises:** [`concat`, `concat_ws`](pyspark-reference-04-functions-strings-and-math.ipynb#concat-concat_ws)

*One missing part should not blank the whole label.*

Return `id` and `display`: title-cased first, middle and last names separated by single spaces, skipping missing parts. Order by `id`.

In [ ]:
names = spark.createDataFrame([(1, "ada", None, "lovelace"), (2, "grace", "brewster", "hopper"), (3, "lin", None, None)], "id int, first string, middle string, last string")
names.show()

**Expected output**

| id | display |
| --- | --- |
| 1 | Ada Lovelace |
| 2 | Grace Brewster Hopper |
| 3 | Lin |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = names.select("id", F.initcap(F.concat_ws(" ", "first", "middle", "last")).alias("display")).orderBy("id")
result.show()

## Exercise 17: Filter with membership and ranges

**Difficulty:** Easy · about 5 min · **Practises:** [`isin`, `between`](pyspark-reference-04-functions-strings-and-math.ipynb#isin-between)

*Most report filters are a status list and a value range.*

Return `order_id`, `status` and `amount` of `paid` or `shipped` orders with an amount between 80 and 150 inclusive, ordered by `order_id`.

In [ ]:
orders.select("order_id", "status", "amount").show()

**Expected output**

| order_id | status | amount |
| --- | --- | --- |
| 1 | paid | 120.00 |
| 2 | shipped | 84.50 |
| 6 | shipped | 149.00 |
| 12 | shipped | 96.40 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = orders.where(F.col("status").isin("paid", "shipped") & F.col("amount").between(80, 150)).select("order_id", "status", "amount").orderBy("order_id")
result.show()

**Next:** look anything up in [PySpark reference 4: functions for strings, math and conditions](pyspark-reference-04-functions-strings-and-math.ipynb), or test yourself across topics with the [PySpark coding challenges](pyspark-coding-challenges.ipynb).